In [ ]:
%load_ext autoreload
%autoreload 2

# Phase 2: Forecasting Model Comparison

In this notebook, we compare 3 forecasting models:
- **MSTL**
- **Baseline** (Exponential Smoothing)
- **N-BEATS** (Neural network model)

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import logging
import warnings
warnings.filterwarnings('ignore')

from src.utils.data_loader import DataLoader
from src.utils.data_cleaner import DataCleaner
from config import DATA_RAW, DATAFILE, TRAIN_WEEKS, TEST_WEEKS, FORECAST_HORIZON, LOOKBACK_WINDOW

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("Imports successful")

## Load Data

In [ ]:
# Load data
loader = DataLoader(DATA_RAW / DATAFILE)
raw_data = loader.load()
print(f"Raw data shape: {raw_data.shape}")
print(f"Summary: {loader.get_summary()}")

In [ ]:
# Clean data
cleaner = DataCleaner(raw_data, frequency='W')
weekly_demand = cleaner.aggregate_by_frequency()
weekly_demand = cleaner.handle_missing_values()
weekly_demand = cleaner.remove_outliers(method='iqr')

# Split into train and test
train, test = cleaner.train_test_split(train_weeks=TRAIN_WEEKS, test_weeks=TEST_WEEKS)

print(f"Train: {len(train)} weeks ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test: {len(test)} weeks ({test.index[0].date()} to {test.index[-1].date()})")
print(f"\nTrain statistics:")
print(train.describe())

## Init and Train Models

In [ ]:
from src.forecasting.mstl_model import MSTLModel
from src.forecasting.baseline_model import BaselineModel
from src.forecasting.nbeats_model import NBeatsModel
from src.forecasting.model_trainer import ModelTrainer

# Init
models = {
    'MSTL': MSTLModel(horizon=FORECAST_HORIZON, lookback=LOOKBACK_WINDOW),
    'Baseline': BaselineModel(horizon=FORECAST_HORIZON, lookback=LOOKBACK_WINDOW),
    'N-BEATS': NBeatsModel(
        horizon=FORECAST_HORIZON,
        lookback=LOOKBACK_WINDOW,
        epochs=50,
        batch_size=32,
        hidden_size=128,
        num_stacks=30,
    )
}

print("Models initialized:")
for name, model in models.items():
    print(f"  - {name}: {model}")

In [ ]:
# Train all models
trainers = {}
training_results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    trainer = ModelTrainer(model, train, test)
    metrics = trainer.train()
    trainers[name] = trainer
    
    result = {
        'Model': name,
        'Training Time (s)': trainer.get_training_time(),
        **metrics
    }
    training_results.append(result)
    print(f"Training completed")
    print(f"    MAE: {metrics['mae']:.2f}")
    print(f"    SMAPE: {metrics['smape']:.2f}%")
    print(f"    RMSE: {metrics['rmse']:.2f}")
    print(f"    Time: {result['Training Time (s)']:.2f}s")

print("\n All models trained successfully")

## Evaluation

In [ ]:
from src.forecasting.model_evaluator import ModelEvaluator

# Evaluate all models
evaluator = ModelEvaluator(models, test)
results_df = evaluator.evaluate_all()

print("\n" + "="*60)
print("MODEL COMPARISON RESULTS")
print("="*60)
print(evaluator.get_comparison_table().to_string(index=False))
print("="*60)

# Print summary
summary = evaluator.get_summary()
print(f"\n  Best Model: {summary['best_model']}")
print(f"   SMAPE: {summary['best_smape']}")
print(f"   MAE: {summary['best_mae']}")
print(f"\nImprovement over worst model: {summary['improvement']}")

## Visualizations

In [ ]:
# Plot SMAPE comparison
fig_smape = evaluator.plot_smape_comparison()
fig_smape.show()
print("SMAPE comparison chart saved")

In [ ]:
# Plot MAE comparison
fig_mae = evaluator.plot_mae_comparison()
fig_mae.show()
print("MAE comparison chart saved")

In [ ]:
# Plot forecast comparison
fig_forecast = evaluator.plot_forecast_comparison(train_data=train)
fig_forecast.show()
print("Forecast comparison chart saved")

In [ ]:
# Plot metrics heatmap
fig_heatmap = evaluator.plot_metrics_heatmap()
fig_heatmap.show()
print("Metrics heatmap saved")

## Findings

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame(training_results)
comparison_df = comparison_df.sort_values('smape')

print("\nDetailed Performance Comparison:")
print(comparison_df.to_string(index=False))

# Rank models
comparison_df['Rank'] = range(1, len(comparison_df) + 1)
comparison_df['SMAPE %'] = comparison_df['smape'].apply(lambda x: f"{x:.2f}%")
comparison_df['MAE'] = comparison_df['mae'].apply(lambda x: f"{x:.2f}")
comparison_df['Training Time'] = comparison_df['Training Time (s)'].apply(lambda x: f"{x:.2f}s")

print("\nRanked Models:")
print(comparison_df[['Rank', 'Model', 'SMAPE %', 'MAE', 'Training Time']].to_string(index=False))

In [ ]:
# Analyze improvements
best_smape = comparison_df['smape'].min()
worst_smape = comparison_df['smape'].max()
improvement_pct = (worst_smape - best_smape) / worst_smape * 100

print(f"\n  Insights:")
print(f"\n1. ACCURACY IMPROVEMENTS:")
print(f"   - Best SMAPE: {best_smape:.2f}% (Best Model)")
print(f"   - Worst SMAPE: {worst_smape:.2f}%")
print(f"   - Improvement: {improvement_pct:.1f}%")

print(f"\n2. TRAINING TIME:")
fastest = comparison_df.loc[comparison_df['Training Time (s)'].idxmin()]
slowest = comparison_df.loc[comparison_df['Training Time (s)'].idxmax()]
print(f"   - Fastest: {fastest['Model']} ({fastest['Training Time (s)']:.2f}s)")
print(f"   - Slowest: {slowest['Model']} ({slowest['Training Time (s)']:.2f}s)")
time_ratio = slowest['Training Time (s)'] / fastest['Training Time (s)']
print(f"   - Ratio: {time_ratio:.1f}x slower")